# Bayesian Epistemic Routing v3: Kaggle smoke test (Qwen3.5-9B, llama.cpp, 2×T4)

**Purpose:** before building the labelling pipeline, check on the real hardware that
1. llama.cpp runs **Qwen3.5-9B Q4_K_M** (the same file you will run locally on the RTX 3050) on a T4,
2. **thinking is off** (answers start immediately, no `<think>` content),
3. the **"I don't know" probes and entity familiarity** behave sensibly (they are signals of the network),
4. **throughput**, giving a real estimate of the labelling time,
5. the **real reranker and QA models** pass the signal tests.

**Settings:** Accelerator **GPU T4 ×2**, Internet **on**.
**Inputs:** upload the `prob_reas/v3` folder as a Kaggle dataset named **`prob-reas-v3`**.
Optional: if you saved the llama.cpp build from an earlier run as a dataset named **`llama-cpp-t4`**, the build step is skipped.

Run all cells. The last cell prints a PASS/FAIL table and writes `smoke_results.json`.

In [ ]:
import os, json, time, subprocess, glob, shutil, sys
CFG = {
    "gguf_repo": "unsloth/Qwen3.5-9B-GGUF",
    "gguf_file": "Qwen3.5-9B-Q4_K_M.gguf",       # same file as the local demo
    "ctx": 16384, "parallel": 4,                    # 4 slots per GPU -> 4096 tokens per slot
    "ports": [8080, 8081],                          # one server per T4
    "n_questions_planned": 11000, "n_action_subset": 2000,
    "build_title_index": False,                     # True: also builds the Wikipedia title index (~10 min)
}
CODE = next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/signals_v3.py", recursive=True)), None)
assert CODE, "add the prob-reas-v3 dataset (the v3 folder) to this notebook"
shutil.copytree(CODE, "/kaggle/working/v3", dirs_exist_ok=True)
sys.path.insert(0, "/kaggle/working/v3")
RESULTS = {"config": CFG, "checks": {}}
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

## 1. llama.cpp built for the T4 (compute capability 7.5)

In [ ]:
BIN = "/kaggle/working/llama-bin"
cached = glob.glob("/kaggle/input/llama-cpp-t4*/llama-server") + glob.glob("/kaggle/input/llama-cpp-t4*/*/llama-server")
if cached:
    shutil.copytree(os.path.dirname(cached[0]), BIN, dirs_exist_ok=True)
    print("using cached build:", cached[0])
else:
    t0 = time.time()
    cmds = [
        "git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/tmp/llama.cpp",
        "cd /kaggle/tmp/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null",
        "cd /kaggle/tmp/llama.cpp && cmake --build build --config Release -j 4 --target llama-server > /kaggle/tmp/build.log 2>&1",
    ]
    for c in cmds:
        r = subprocess.run(c, shell=True)
        assert r.returncode == 0, f"failed: {c} (see /kaggle/tmp/build.log)"
    shutil.copytree("/kaggle/tmp/llama.cpp/build/bin", BIN, dirs_exist_ok=True)
    print(f"built in {(time.time() - t0) / 60:.1f} min")
commit = subprocess.run("cd /kaggle/tmp/llama.cpp && git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
RESULTS["llama_cpp_commit"] = commit or "cached build (see the dataset that holds it)"
print("llama.cpp commit:", RESULTS["llama_cpp_commit"])

## 2. The model file

In [ ]:
from huggingface_hub import hf_hub_download
t0 = time.time()
MODEL = hf_hub_download(CFG["gguf_repo"], CFG["gguf_file"], local_dir="/kaggle/tmp/models")
print(MODEL, f"{os.path.getsize(MODEL) / 1e9:.2f} GB in {time.time() - t0:.0f}s")

## 3. Two servers, one per GPU

In [ ]:
import urllib.request
def healthy(port):
    try:
        return json.loads(urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2).read()).get("status") == "ok"
    except Exception:
        return False

def start(gpu, port, extra):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "LD_LIBRARY_PATH": BIN + ":" + os.environ.get("LD_LIBRARY_PATH", "")}
    cmd = [f"{BIN}/llama-server", "-m", MODEL, "-ngl", "99", "-c", str(CFG["ctx"]), "--parallel", str(CFG["parallel"]),
           "--port", str(port), "--host", "127.0.0.1", "--jinja"] + extra
    log = open(f"/kaggle/tmp/server{gpu}.log", "w")
    return subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)

# thinking off at server level if this llama.cpp supports the flag; our probes render the
# non-thinking template themselves, so they work either way
FLAG_SETS = [["--chat-template-kwargs", '{"enable_thinking": false}'], ["--reasoning-budget", "0"], []]
SERVERS = []
for gpu, port in enumerate(CFG["ports"]):
    for flags in FLAG_SETS:
        p = start(gpu, port, flags)
        for _ in range(180):
            if healthy(port) or p.poll() is not None:
                break
            time.sleep(1)
        if healthy(port):
            SERVERS.append((p, port, flags)); print(f"GPU{gpu}: server on :{port} with flags {flags}"); break
        p.kill(); print(f"GPU{gpu}: flags {flags} failed, trying the next set")
    else:
        print(open(f"/kaggle/tmp/server{gpu}.log").read()[-3000:])
        raise RuntimeError("server did not start")
RESULTS["server_flags"] = SERVERS[0][2]
print(subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv", shell=True, capture_output=True, text=True).stdout)

## 4. Check: thinking is off, answers are short

In [ ]:
from llm_probe_v3 import LlamaServer, render_qwen, rag_prompt, clean, IDK
S0 = LlamaServer(f"http://127.0.0.1:{CFG['ports'][0]}")
qs = ["What is the capital of France?", "Who wrote the novel Pride and Prejudice?", "In what year did the Berlin Wall fall?"]
think_leak, outs = False, []
for q in qs:
    r = S0.complete(render_qwen(f"Answer in a few words.\n\nQuestion: {q}"), n_predict=32, stop=("<|im_end|>",))
    txt = r.get("content", "")
    think_leak |= "<think>" in txt or "</think>" in txt
    outs.append(txt.strip())
    print(f"{q:<45} -> {txt.strip()[:80]!r}")
# the template the server itself would render, for comparison
try:
    t = S0._post("/apply-template", {"messages": [{"role": "user", "content": "hi"}]})["prompt"]
    print("\nserver template:", repr(t)); print("our template:   ", repr(render_qwen("hi")))
    RESULTS["server_template"] = t
except Exception as e:
    print("(no /apply-template endpoint)", e)
ok = (not think_leak) and all(len(o.split()) <= 12 for o in outs)
RESULTS["checks"]["thinking_off_short_answers"] = ok
print("\nPASS" if ok else "\nFAIL: thinking content or long answers - check the template above")

## 5. Check: the LLM signals behave sensibly
Expected: P(IDK) **low** with a helpful passage, **high** with an unrelated one; closed-book P(IDK) **lower** for a famous fact than an obscure one; entity log-prob **higher** for a famous entity.

In [ ]:
helpful = ["Inception is a 2010 science fiction film written and directed by Christopher Nolan."]
useless = ["A banana is an elongated, edible fruit produced by several kinds of large herbaceous flowering plants."]
q = "Who directed Inception?"
t0 = time.time()
g_help = S0.grounded_p_idk(q, helpful); g_use = S0.grounded_p_idk("Who was the screenwriter for Kuttram Kadithal?", useless)
cb_fam = S0.closed_book("What is the capital of France?"); cb_obs = S0.closed_book("Who was the father of Jitendra Narayan?")
e_fam = S0.entity_logprob("Paris"); e_obs = S0.entity_logprob("Nagathihalli Chandrashekhar")
probe_secs = time.time() - t0
rows = [("grounded P(IDK), helpful passage", g_help.prob), ("grounded P(IDK), unrelated passage", g_use.prob),
        ("closed-book P(IDK), famous fact", cb_fam[1].prob), ("closed-book P(IDK), obscure fact", cb_obs[1].prob),
        ("entity mean log-prob, Paris", e_fam.mean), ("entity mean log-prob, obscure name", e_obs.mean)]
for name, v in rows: print(f"{name:<40} {v:9.4f}")
print("closed-book answers:", cb_fam[0], "|", cb_obs[0])
print(f"floored tokens (outside top-100): {g_help.floored + g_use.floored + cb_fam[1].floored + cb_obs[1].floored}")
ok = g_help.prob < g_use.prob and cb_fam[1].prob < cb_obs[1].prob and e_fam.mean > e_obs.mean
RESULTS["checks"]["llm_signals_sensible"] = ok
RESULTS["probe_values"] = {n: v for n, v in rows}
print("PASS" if ok else "FAIL: inspect the values above")

## 6. Throughput and the labelling-time estimate
Runs the full per-question LLM work (answer + closed-book answer + grounded P(IDK) + closed-book P(IDK) + entity familiarity)
on ~1,200-token prompts, 4 workers per GPU. Note: Qwen3.5 is a hybrid (Gated DeltaNet) model; if the
teacher-forcing steps cannot reuse the cached prefix, they will be slow here, and this test shows it.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
FILLER = ("The river flows through the northern plains before reaching the coast, where the old harbour town was founded "
          "by merchants in the fourteenth century. Its economy depends on fishing, tourism and a small shipyard. ") * 6
def bundle(i):
    srv = LlamaServer(f"http://127.0.0.1:{CFG['ports'][i % 2]}")
    slot = (i // 2) % CFG["parallel"]
    passages = [f"Passage {k} about topic {i}: " + FILLER for k in range(3)]
    q = f"Who founded the harbour town described in passage {i % 3}?"
    t0 = time.time()
    ans, timing = srv.answer(q, passages, slot=slot)
    t2 = srv.tier2(q, passages, ["Harbour Town"], slot=slot)
    return time.time() - t0, timing

N = 32
t0 = time.time()
with ThreadPoolExecutor(max_workers=CFG["parallel"] * 2) as ex:
    res = list(ex.map(bundle, range(N)))
wall = time.time() - t0
pp = [r[1].get("prompt_per_second") for r in res if r[1].get("prompt_per_second")]
tg = [r[1].get("predicted_per_second") for r in res if r[1].get("predicted_per_second")]
per_q = wall / N
plan = CFG["n_questions_planned"] + 0.6 * CFG["n_action_subset"] * 4      # 4 extra actions, ~60% of a bundle each
est_h = plan * per_q / 3600
print(f"{N} full question bundles in {wall:.0f}s -> {per_q:.2f}s per question (both GPUs together)")
print(f"prompt processing ~{sum(pp) / max(len(pp), 1):.0f} tok/s per request, generation ~{sum(tg) / max(len(tg), 1):.1f} tok/s per request")
print(f"labelling estimate for {CFG['n_questions_planned']} questions + action subset: ~{est_h:.1f} GPU-hours (wall clock on one 2xT4 session)")
RESULTS["throughput"] = {"seconds_per_question": per_q, "estimated_hours": est_h,
                         "prompt_tok_s": pp[:5], "gen_tok_s": tg[:5]}
RESULTS["checks"]["labelling_fits_kaggle_quota"] = est_h < 3 * 30   # three accounts x 30 GPU-hours/week

## 7. Signal tests with the REAL reranker and QA models

In [ ]:
subprocess.run("pip install -q spacy rapidfuzz && python -m spacy download en_core_web_sm -q", shell=True)
r = subprocess.run("cd /kaggle/working/v3 && python test_signals_v3.py --models", shell=True, capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
RESULTS["checks"]["signal_tests_real_models"] = r.returncode == 0

## 8. (optional) Wikipedia title index for NameCollision

In [ ]:
if CFG["build_title_index"]:
    r = subprocess.run("cd /kaggle/working/v3 && python build_title_index.py --out /kaggle/working/title_index.tsv.gz",
                       shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-1000:])
    RESULTS["checks"]["title_index_built"] = r.returncode == 0
else:
    print("skipped (set CFG['build_title_index'] = True)")

## Summary

In [ ]:
for p, _, _ in SERVERS:
    p.kill()
json.dump(RESULTS, open("/kaggle/working/smoke_results.json", "w"), indent=1, default=str)
print(f"{'check':<36}result")
for k, v in RESULTS["checks"].items():
    print(f"{k:<36}{'PASS' if v else 'FAIL'}")
print("\nSave this notebook's output as a dataset named 'llama-cpp-t4' to reuse the llama.cpp build (llama-bin/).")
print("Send smoke_results.json back for the next step.")